**Prérequis : `02_nettoyage_silver.ipynb`** exécuté (Silver disponible dans `data/silver/`).

---

# TP3-5 — Anti-fuite, feature engineering et constitution de la couche Gold

**Cas d'usage :** prédiction du churn — éditeur SaaS B2B.

**Portée de ce notebook.** Partir de la couche **Silver** (`data/silver/`, produite
par `02_nettoyage_silver.ipynb`) — table propre, typée, dédoublonnée. Ce notebook
produit la couche **Gold** : le périmètre de features du modèle de churn, débarrassé
du piège de fuite, enrichi de deux features dérivées, et séparé en train/test.
**Il s'arrête à la donnée prête à l'entraînement** — l'entraînement lui-même
(baseline, comparaison de modèles, seuil) est hors périmètre de ce notebook.

In [1]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

SILVER_DIR = Path("../data/silver")
GOLD_DIR = Path("../data/gold")
GOLD_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_AT = datetime.now(timezone.utc).isoformat()
RANDOM_STATE = 42
LAYER_VERSION = "v1"


def sha256_of(path: Path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

clients = pd.read_parquet(SILVER_DIR / "clients_churn_silver.parquet")
print(f"clients (Silver) : {clients.shape[0]} lignes x {clients.shape[1]} colonnes")
print("Taux de churn :", round(100 * clients["churn"].mean(), 1), "%")

clients (Silver) : 5000 lignes x 34 colonnes
Taux de churn : 28.0 %


## §1 — Le piège de fuite : `sante_compte_fin_periode`

Le nom de cette variable est en soi un signal : un score de santé calculé **en fin de
période** est mesuré après la fenêtre d'observation, donc après la décision que le
modèle est censé anticiper. On quantifie l'ampleur du problème avant de trancher.

In [2]:
corr_fuite = clients["sante_compte_fin_periode"].astype(float).corr(clients["churn"].astype(float))
print(f"Corrélation sante_compte_fin_periode / churn : {corr_fuite:.3f}")

# Démonstration : un modèle entraîné sur cette seule variable atteint-il une AUC suspecte ?
X_demo = clients[["sante_compte_fin_periode"]]
y_demo = clients["churn"]
modele_demo = LogisticRegression().fit(X_demo, y_demo)
proba_demo = modele_demo.predict_proba(X_demo)[:, 1]
auc_demo = roc_auc_score(y_demo, proba_demo)
print(f"ROC-AUC d'un modèle à 1 seule variable (sante_compte_fin_periode) : {auc_demo:.3f}")
print()
print("=> Une AUC aussi élevée avec une seule variable est exactement le signal d'alerte")
print("   décrit dans l'énoncé (§6) : à exclure, pas à célébrer.")

Corrélation sante_compte_fin_periode / churn : -0.881
ROC-AUC d'un modèle à 1 seule variable (sante_compte_fin_periode) : 0.999

=> Une AUC aussi élevée avec une seule variable est exactement le signal d'alerte
   décrit dans l'énoncé (§6) : à exclure, pas à célébrer.


## §2 — Les leurres : test statistique par variable

Certaines variables sont volontairement sans pouvoir prédictif. On les repère par un
test statistique adapté au type de variable, plutôt qu'à l'œil :
- **Numériques** : corrélation point-bisériale avec `churn` (équivalente à Pearson sur
  une cible binaire).
- **Catégorielles** : test du χ² d'indépendance + V de Cramér (taille d'effet — un
  p-value significatif avec un grand échantillon ne veut pas dire grand-chose seul).

In [3]:
NUMERIC_CANDIDATES = [
    "anciennete_mois", "sieges_souscrits", "utilisateurs_actifs", "taux_adoption_pct",
    "connexions_30j", "heures_usage_30j", "fonctionnalites_total", "fonctionnalites_utilisees",
    "nb_integrations", "derniere_connexion_jours", "tickets_support_90j",
    "delai_reponse_support_h", "csat", "retards_paiement_12m", "revenu_mensuel_recurrent_eur",
]

resultats_num = []
for col in NUMERIC_CANDIDATES:
    r, p = stats.pointbiserialr(clients["churn"], clients[col].astype(float))
    resultats_num.append({"variable": col, "correlation": round(r, 3), "p_value": round(p, 4)})

resultats_num_df = pd.DataFrame(resultats_num).sort_values("correlation", key=abs, ascending=False)
resultats_num_df

,variable,correlation,p_value
8,nb_integrations,-0.364,0.0000
9,derniere_connexion_jours,0.353,0.0000
0,anciennete_mois,-0.325,0.0000
3,taux_adoption_pct,-0.297,0.0000
13,retards_paiement_12m,0.296,0.0000
10,tickets_support_90j,0.270,0.0000
12,csat,-0.265,0.0000
7,fonctionnalites_utilisees,-0.257,0.0000
11,delai_reponse_support_h,0.239,0.0000
4,connexions_30j,-0.237,0.0000


In [4]:
CATEGORICAL_CANDIDATES = [
    "secteur", "pays", "taille_entreprise", "plan", "jour_souscription",
    "couleur_theme_interface", "code_datacenter", "groupe_experimentation",
]

def cramers_v(confusion: np.ndarray) -> float:
    chi2 = stats.chi2_contingency(confusion)[0]
    n = confusion.sum()
    k = min(confusion.shape) - 1
    return float(np.sqrt(chi2 / (n * k))) if k > 0 else float("nan")


resultats_cat = []
for col in CATEGORICAL_CANDIDATES:
    table = pd.crosstab(clients[col], clients["churn"])
    chi2, p, _, _ = stats.chi2_contingency(table)
    v = cramers_v(table.values)
    resultats_cat.append({"variable": col, "cramers_v": round(v, 3), "p_value": round(p, 4)})

resultats_cat_df = pd.DataFrame(resultats_cat).sort_values("cramers_v", ascending=False)
resultats_cat_df

,variable,cramers_v,p_value
3,plan,0.138,0.0000
2,taille_entreprise,0.068,0.0000
0,secteur,0.052,0.0577
4,jour_souscription,0.045,0.1212
1,pays,0.035,0.3980
6,code_datacenter,0.019,0.6343
5,couleur_theme_interface,0.015,0.8862
7,groupe_experimentation,0.004,0.9658


In [5]:
SEUIL_CORR = 0.03
SEUIL_CRAMER = 0.05

leurres_num = resultats_num_df.loc[resultats_num_df["correlation"].abs() < SEUIL_CORR, "variable"].tolist()
leurres_cat = resultats_cat_df.loc[resultats_cat_df["cramers_v"] < SEUIL_CRAMER, "variable"].tolist()
leurres = leurres_num + leurres_cat

print(f"Leurres identifiés (effet quasi nul, seuils corr<{SEUIL_CORR} / Cramér's V<{SEUIL_CRAMER}) :")
for v in leurres:
    print(" -", v)
print()
print("=> Ces variables restent dans le jeu de features (§3) : l'objectif n'est pas de les")
print("   retirer, mais de ne pas sur-interpréter leur importance lors de la lecture du")
print("   modèle (permutation importance, à faire dans le notebook de modélisation).")

Leurres identifiés (effet quasi nul, seuils corr<0.03 / Cramér's V<0.05) :
 - sieges_souscrits
 - jour_souscription
 - pays
 - code_datacenter
 - couleur_theme_interface
 - groupe_experimentation

=> Ces variables restent dans le jeu de features (§3) : l'objectif n'est pas de les
   retirer, mais de ne pas sur-interpréter leur importance lors de la lecture du
   modèle (permutation importance, à faire dans le notebook de modélisation).


## §3 — Décision : colonnes exclues de la modélisation

Trois catégories d'exclusion, chacune pour une raison différente — à ne pas confondre
avec les leurres (§2), qui restent des features valides.

In [6]:
exclusions = pd.DataFrame([
    {"colonne": "client_id", "raison": "Identifiant, aucune valeur prédictive par construction"},
    {"colonne": "date_souscription", "raison": "Brute, déjà représentée par anciennete_mois et jour_souscription"},
    {"colonne": "commentaire_csm", "raison": "Texte libre, pas une feature directe (décision actée en Silver)"},
    {"colonne": "sante_compte_fin_periode", "raison": "PIÈGE DE FUITE — calculée après la période observée (§1)"},
    {"colonne": "valeur_vie_client_eur", "raison": "Cible secondaire (régression) — ne doit jamais nourrir le modèle principal"},
    {"colonne": "churn", "raison": "Cible principale — séparée en y, pas dans X"},
])
exclusions

,colonne,raison
0,client_id,"Identifiant, aucune valeur prédictive par cons..."
1,date_souscription,"Brute, déjà représentée par anciennete_mois et..."
2,commentaire_csm,"Texte libre, pas une feature directe (décision..."
3,sante_compte_fin_periode,PIÈGE DE FUITE — calculée après la période obs...
4,valeur_vie_client_eur,Cible secondaire (régression) — ne doit jamais...
5,churn,"Cible principale — séparée en y, pas dans X"


## §4 — Feature engineering léger (ratios)

Deux ratios simples, directement lisibles par le métier — pas une explosion de
features dérivées non justifiées.

In [7]:
clients["taux_utilisation_fonctionnalites"] = (
    clients["fonctionnalites_utilisees"] / clients["fonctionnalites_total"].replace(0, np.nan)
).fillna(0)

clients["taux_retard_paiement_par_mois"] = (
    clients["retards_paiement_12m"] / clients["anciennete_mois"].clip(lower=1)
)

print(clients[["taux_utilisation_fonctionnalites", "taux_retard_paiement_par_mois"]].describe())

       taux_utilisation_fonctionnalites  taux_retard_paiement_par_mois
count                       5000.000000                    5000.000000
mean                           0.334749                       0.367359
std                            0.188917                       0.905333
min                            0.000000                       0.000000
25%                            0.200000                       0.000000
50%                            0.350000                       0.000000
75%                            0.475000                       0.200000
max                            1.000000                       7.000000


In [8]:
colonnes_exclues = [
    "client_id", "date_souscription", "commentaire_csm",
    "sante_compte_fin_periode", "valeur_vie_client_eur", "churn",
]
feature_columns = [c for c in clients.columns if c not in colonnes_exclues]

print(f"{len(feature_columns)} features retenues pour le modèle principal :")
for c in feature_columns:
    marker = " (leurre identifié §2)" if c in leurres else ""
    print(f" - {c}{marker}")

30 features retenues pour le modèle principal :
 - jour_souscription (leurre identifié §2)
 - secteur
 - pays (leurre identifié §2)
 - taille_entreprise
 - plan
 - anciennete_mois
 - sieges_souscrits (leurre identifié §2)
 - utilisateurs_actifs
 - taux_adoption_pct
 - connexions_30j
 - heures_usage_30j
 - fonctionnalites_total
 - fonctionnalites_utilisees
 - nb_integrations
 - derniere_connexion_jours
 - tickets_support_90j
 - delai_reponse_support_h
 - csat
 - retards_paiement_12m
 - revenu_mensuel_recurrent_eur
 - couleur_theme_interface (leurre identifié §2)
 - code_datacenter (leurre identifié §2)
 - groupe_experimentation (leurre identifié §2)
 - prix_mensuel_par_siege_eur
 - fonctionnalites_incluses
 - sla_reponse_h
 - quota_stockage_go
 - support_dedie
 - taux_utilisation_fonctionnalites
 - taux_retard_paiement_par_mois


## §5 — Split train/test stratifié

Même logique que la comparaison de modèles à venir : 80/20, stratifié sur `churn`
pour préserver le taux de churn dans les deux ensembles (déséquilibre 72/28).

In [9]:
train_idx, test_idx = train_test_split(
    clients.index,
    test_size=0.2,
    stratify=clients["churn"],
    random_state=RANDOM_STATE,
)

clients["split"] = "train"
clients.loc[test_idx, "split"] = "test"

recap_split = clients.groupby("split").agg(
    lignes=("churn", "size"),
    taux_churn_pct=("churn", lambda s: round(100 * s.mean(), 1)),
)
recap_split

,lignes,taux_churn_pct
split,,
test,1000,28.0
train,4000,28.0


## §6 — Persistance de la couche Gold

Une seule table, avec une colonne `split` figée et versionnée — traçable et
reproductible, plutôt qu'un `train_test_split` invisible relancé à chaque exécution
du notebook de modélisation avec un risque de seed oublié.

In [10]:
colonnes_gold = ["client_id", "split"] + feature_columns + ["sante_compte_fin_periode", "valeur_vie_client_eur", "churn"]
gold = clients[colonnes_gold].copy()

gold_path = GOLD_DIR / "clients_churn_gold.parquet"
gold.to_parquet(gold_path, index=False)

manifest = {
    "couche": "gold",
    "version": LAYER_VERSION,
    "processed_at_utc": PROCESSED_AT,
    "source": "data/silver/clients_churn_silver.parquet",
    "sha256_source_silver": sha256_of(SILVER_DIR / "clients_churn_silver.parquet"),
    "lignes": int(len(gold)),
    "colonnes": list(gold.columns),
    "features_modele_principal": feature_columns,
    "leurres_identifies": leurres,
    "colonnes_exclues_du_modele": colonnes_exclues,
    "piege_de_fuite": {
        "colonne": "sante_compte_fin_periode",
        "correlation_avec_churn": round(float(corr_fuite), 3),
        "auc_modele_1_variable": round(float(auc_demo), 3),
        "conservee_dans_gold_pour": "traçabilité et régression CLV -- exclue du X du modèle churn (voir §3)",
    },
    "cible_secondaire": "valeur_vie_client_eur (régression, jamais utilisée comme feature du modèle principal)",
    "split": {
        "méthode": "train_test_split stratifié sur churn",
        "random_state": RANDOM_STATE,
        "test_size": 0.2,
        "taux_churn_train_pct": float(recap_split.loc["train", "taux_churn_pct"]),
        "taux_churn_test_pct": float(recap_split.loc["test", "taux_churn_pct"]),
    },
    "chemin_gold": str(gold_path),
    "sha256_gold": sha256_of(gold_path),
}
manifest_path = GOLD_DIR / "gold_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("Couche Gold écrite :", gold_path)
print("Manifeste écrit    :", manifest_path)

Couche Gold écrite : ..\data\gold\clients_churn_gold.parquet
Manifeste écrit    : ..\data\gold\gold_manifest.json


## §7 — Vérification finale

In [11]:
verif = pd.DataFrame([
    {"vérification": "Lignes", "résultat": len(gold)},
    {"vérification": "Colonnes", "résultat": gold.shape[1]},
    {"vérification": "Features du modèle principal", "résultat": len(feature_columns)},
    {"vérification": "Lignes train", "résultat": int((gold["split"] == "train").sum())},
    {"vérification": "Lignes test", "résultat": int((gold["split"] == "test").sum())},
    {"vérification": "sante_compte_fin_periode absente de X", "résultat": "sante_compte_fin_periode" not in feature_columns},
    {"vérification": "valeur_vie_client_eur absente de X", "résultat": "valeur_vie_client_eur" not in feature_columns},
])
verif

,vérification,résultat
0,Lignes,5000
1,Colonnes,35
2,Features du modèle principal,30
3,Lignes train,4000
4,Lignes test,1000
5,sante_compte_fin_periode absente de X,True
6,valeur_vie_client_eur absente de X,True


## §8 — Vue de features pour la régression CLV (cible secondaire)

`valeur_vie_client_eur` est la cible d'un second modèle (régression), distinct du modèle churn. Les colonnes `client_id`, `date_souscription` et `commentaire_csm` ont déjà été retirées de Gold au moment de la construction de `feature_columns` (§3) ; il reste à trancher le sort de `sante_compte_fin_periode` et de `churn` au sein des colonnes disponibles dans Gold, pour cette seconde cible.

**Principe retenu** : une variable calculée en fin de période n'est pas disponible au moment du scoring, quel que soit le modèle qui la consommerait -- le principe anti-fuite s'applique à la disponibilité de l'information, pas seulement à sa force de corrélation avec une cible particulière. `sante_compte_fin_periode` et `churn` (l'issue elle-même, inconnue au moment du scoring) sont donc exclues de `X_clv` par le même raisonnement temporel que pour `X_churn` -- indépendamment de leur corrélation mesurée avec `valeur_vie_client_eur`.

In [12]:
corr_sante_clv = clients["sante_compte_fin_periode"].astype(float).corr(clients["valeur_vie_client_eur"].astype(float))
corr_sante_log_clv = clients["sante_compte_fin_periode"].astype(float).corr(np.log1p(clients["valeur_vie_client_eur"].astype(float)))
print(f"Corrélation sante_compte_fin_periode / valeur_vie_client_eur : {corr_sante_clv:.3f}")
print(f"Corrélation sante_compte_fin_periode / log(valeur_vie_client_eur) : {corr_sante_log_clv:.3f}")
print(f"(rappel, sante_compte_fin_periode / churn : {corr_fuite:.3f})")
print()
print("=> Corrélation nettement plus faible qu'avec churn : ce n'est pas un piège de fuite")
print("   au sens statistique pour cette cible. La décision ci-dessous repose néanmoins sur")
print("   la disponibilité de l'information au moment du scoring, pas sur ce chiffre --")
print("   elle serait exclue même si la corrélation avait été forte.")

Corrélation sante_compte_fin_periode / valeur_vie_client_eur : 0.112
Corrélation sante_compte_fin_periode / log(valeur_vie_client_eur) : 0.187
(rappel, sante_compte_fin_periode / churn : -0.881)

=> Corrélation nettement plus faible qu'avec churn : ce n'est pas un piège de fuite
   au sens statistique pour cette cible. La décision ci-dessous repose néanmoins sur
   la disponibilité de l'information au moment du scoring, pas sur ce chiffre --
   elle serait exclue même si la corrélation avait été forte.


In [13]:
exclusions_clv = pd.DataFrame([
    {"colonne": "client_id", "raison": "Déjà absente de Gold (identifiant, retirée §3)"},
    {"colonne": "date_souscription", "raison": "Déjà absente de Gold (retirée §3)"},
    {"colonne": "commentaire_csm", "raison": "Déjà absente de Gold (retirée §3)"},
    {"colonne": "sante_compte_fin_periode", "raison": "Calculée en fin de période -- non disponible au moment du scoring, quelle que soit sa corrélation avec cette cible"},
    {"colonne": "churn", "raison": "Issue non connue au moment du scoring -- même contrainte temporelle que sante_compte_fin_periode"},
    {"colonne": "valeur_vie_client_eur", "raison": "Cible de ce modèle -- séparée en y, pas dans X"},
])
exclusions_clv

,colonne,raison
0,client_id,"Déjà absente de Gold (identifiant, retirée §3)"
1,date_souscription,Déjà absente de Gold (retirée §3)
2,commentaire_csm,Déjà absente de Gold (retirée §3)
3,sante_compte_fin_periode,Calculée en fin de période -- non disponible a...
4,churn,Issue non connue au moment du scoring -- même ...
5,valeur_vie_client_eur,"Cible de ce modèle -- séparée en y, pas dans X"


In [14]:
colonnes_exclues_clv = ["client_id", "split", "sante_compte_fin_periode", "churn", "valeur_vie_client_eur"]
feature_columns_clv = [c for c in gold.columns if c not in colonnes_exclues_clv]

print(f"{len(feature_columns_clv)} features retenues pour la régression CLV")
print("Identique à X_churn :", set(feature_columns_clv) == set(feature_columns))

30 features retenues pour la régression CLV
Identique à X_churn : True


In [15]:
verif_clv = pd.DataFrame([
    {"vérification": "Features retenues (X_clv)", "résultat": len(feature_columns_clv)},
    {"vérification": "Identique à X_churn", "résultat": set(feature_columns_clv) == set(feature_columns)},
    {"vérification": "sante_compte_fin_periode absente de X_clv", "résultat": "sante_compte_fin_periode" not in feature_columns_clv},
    {"vérification": "churn absente de X_clv", "résultat": "churn" not in feature_columns_clv},
])
verif_clv

,vérification,résultat
0,Features retenues (X_clv),30
1,Identique à X_churn,True
2,sante_compte_fin_periode absente de X_clv,True
3,churn absente de X_clv,True


In [16]:
with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

manifest["piege_de_fuite"]["conservee_dans_gold_pour"] = (
    "traçabilité et régression CLV -- exclue du X des deux modèles (churn §3, CLV §8) "
    "pour la même raison temporelle, pas seulement pour le churn"
)
manifest["cible_secondaire_features"] = {
    "colonnes_exclues_clv": colonnes_exclues_clv,
    "feature_columns_clv": feature_columns_clv,
    "identique_a_feature_columns_modele_principal": set(feature_columns_clv) == set(feature_columns),
    "correlation_sante_compte_fin_periode_vs_clv": round(float(corr_sante_clv), 3),
    "correlation_sante_compte_fin_periode_vs_log_clv": round(float(corr_sante_log_clv), 3),
    "principe": "variable calculée en fin de période exclue par principe de disponibilité au scoring, indépendamment de la force de corrélation mesurée",
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("Manifeste mis à jour :", manifest_path)

Manifeste mis à jour :

 ..\data\gold\gold_manifest.json


## Journal de bord — Synthèse TP3-5 (Gold)

- **Piège de fuite confirmé.** `sante_compte_fin_periode` fortement corrélée au
  churn ; un modèle à une seule variable atteint une AUC très élevée (voir §1) --
  exactement le signal d'alerte décrit dans l'énoncé. Exclue du modèle principal,
  conservée dans Gold pour traçabilité.
- **Leurres identifiés statistiquement**, pas à l'œil -- corrélation point-bisériale
  pour le numérique, V de Cramér pour le catégoriel (§2). Conservés comme features :
  la discipline attendue est de ne pas les sur-interpréter, pas de les supprimer.
- **3 exclusions actées** pour 3 raisons distinctes (identifiant, texte libre, cible
  secondaire) -- à ne pas confondre avec les leurres.
- **Feature engineering minimal** : 2 ratios directement interprétables, pas une
  explosion de variables dérivées.
- **Split reproductible** : train/test 80/20 stratifié, seed fixé et documenté,
  matérialisé par une colonne `split` dans la table Gold plutôt que recalculé à
  chaque exécution.
- **Vue de features dédiée à la régression CLV (§8).** Même principe anti-fuite appliqué à `valeur_vie_client_eur` qu'à `churn` : exclusion de `sante_compte_fin_periode` et de `churn` par disponibilité au scoring, pas par corrélation -- `X_clv` coïncide avec `feature_columns` (30 variables), seule la cible `y` change.
- **Ce notebook s'arrête ici.** L'entraînement (baseline, comparaison de modèles,
  choix du seuil) est le sujet du notebook suivant.